# Numerički integratori i grafovske neuronske mreže za simulaciju dinamike čestica
**Projekat: NANS (Numerički Algoritmi i Numerički Softver)**  
**Platforma/Prezentacija: Interaktivni Python Notebook**

Ova prezentacija predstavlja projekat za ocenu na kursu NANS, a funkcioniše i kao interaktivni uvod u projekat namenjen za GitHub repozitorijum.

## 1. Motivacija i Uvod

Simulacija dinamike čestica predstavlja ključno područje u računarskoj fizici. Tradicionalno se koriste **numerički integratori** kako bi se rešile diferencijalne jednačine kretanja iterativno kroz vreme. Ovi algoritmi su fizički utemeljeni, ali pate od velike osetljivosti na vremenski korak (`dt`) i izuzetno su računarski zahtevni.

Glavni cilj projekta je upoređivanje klasičnog numeričkog pristupa sa pristupom mašinskog učenja pomoću **Grafovskih Neuronskih Mreža (GNN)**, koje omogućavaju bržu simulaciju kompleksne fizike na osnovu prethodnog učenja interakcija iz generisanih podataka.


## 2. NANS u Fokusu: Numerička Simulacija (WCSPH)

Kao osnovu projekta i primarni izvor fizički potpuno korektnih podataka, razvili smo sopstveni **WCSPH (Weakly Compressible Smoothed Particle Hydrodynamics)** softver, umesto oslanjanja samo na jednostavne elastične sile na jednoj dimenziji prostora.

### Matematički Model
Dinamika tečnosti aproksimira se metodom čestica čije interakcije definiše SPH kernel funkcija $W(r, h)$, gde je $h$ radijus uticaja. Centralne jednačine obuhvataju:

1. **Gustina:** $\rho_i = \sum_j m_j W(r_{ij}, h)$  
2. **Pritisak fluida (Taitova jednačina stišljive tečnosti):** $P_i = k \left( \left(\frac{\rho_i}{\rho_0}\right)^\gamma - 1 \right)$
3. **Sila pritiska (gradijent pritiska SPH):** $\mathbf{F}_i^{press} = -\sum_j m_j \left( \frac{P_i}{\rho_i^2} + \frac{P_j}{\rho_j^2} \right) \nabla W(r_{ij}, h)$
4. **Sila viskoznosti:** $\mathbf{F}_i^{visc} = \mu \sum_j m_j \frac{\mathbf{v}_j - \mathbf{v}_i}{\rho_j} \nabla^2 W(r_{ij}, h)$

### Numerički Integrator u Simulaciji
Ključno za veran proračun numeričke simulacije iz projektnog zadatka je izbor efikasnog numeričkog integratora. Iako su podržani **Forward Euler** i **Runge-Kutta 4 (RK4)**, za neophodnu energetsku stabilnost WCSPH sistema implementirali smo varijantu **Symplectic Euler** integratora (koji je uspešno i računarski lakše zamenio projektom inicijalno predloženi Velocity Verlet metod):

$$ \mathbf{v}_{t+1} = \mathbf{v}_t + \mathbf{a}_t \Delta t $$
$$ \mathbf{x}_{t+1} = \mathbf{x}_t + \mathbf{v}_{t+1} \Delta t $$

Simplektički Euler obezbeđuje očuvanje energije faznog prostora duž veoma dugačkih trajektorija simulacije fluida uz izrazito niske računske zahteve.


In [1]:
# Učitavamo okruženje i relevantne simulacione biblioteke.
import sys
import os
import torch
import numpy as np
import yaml
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# Usmeravanje ka korenskom direktorijumu ukoliko je pokrenuto iz presentacionog
if os.getcwd().split('/')[-1] == 'presentation':
    os.chdir('..')
sys.path.append(os.getcwd())

from simulation import FluidSimulation
from models import GNNSimulator
from dataset import build_graph

print("✓ Uspešno importovani paketi klasicnog numeričkog WCSPH SPH integratora i GNN modela projekta!")


✓ Uspešno importovani paketi klasicnog numeričkog WCSPH SPH integratora i GNN modela projekta!


## 3. Skupovi podataka i inovativni trening (Dataset generation i Fine-Tuning)

Učenje ovog modela nije radikalno svelo prepoznavanje tečnosti na jedan uski domen usred oskudice podataka, već se zasniva na pre-training/fine-tuning ideji:

1. **Osnovno učenje (Pre-training):** Naš model započinje razumevanje sveta treniranjem na obimnom **DeepMind WaterDrop datasetu**. Time mreža stiče robusno univerzalno razumevanje ograničenja kontejnera i dinamika kapljica.
2. **Proprietary Generisanje Skupa Podataka (Fine-Tuning):** Pomoću *izvornog Python koda i implementiranih integratora za simulacije na ovom kursu*, generisali smo naš **proprietary WCSPH fluid dataset**. Mreža se putem transfernog učenja (Transfer Learning) na finom dezenu navikla na našu varijantu numerike.


## 4. Arhitektura GNN Modela (Encode - Process - Decode)

Kinetički podaci generisani našom SPH metodom posmatraju se kao podaci na grafu pre nego što se ubace u model. Fizičko stanje fluida posmatramo kroz dinamičan *Radius Graph*:
- Svaka čestica fluida i granica bazena je **čvor**. Interakcije fluida u izabranom domenu dejstva sila (radijusu dejstva jezgra $h$) postaju **veze/grane (edges)**.
- **Encoder (Kodiranje):** Prvo projektuje prethodne položaje i brzine čestica u latentni (skriveni) matematički prostor dimenzionalnosti $128$ koristeći klasični Multilayer Perceptron.
- **Process (Procesiranje Pritiska):** Deluje kao serija InteractionNetwork algoritama posvećenih prolasku poruka (Message-Passing arhitekture), koji simuliraju razmenu sila sa susednim partiklima.
- **Decode (Dekodiranje Stanja):** Nakon niza iteracija razmene i konvolucije po partiklu grafa, skriveno grafovsko stanje nazad prevodimo u vektor pritiska, odnosno izvedeno predviđeno **ubrzanje** vektora brzine $\mathbf{a}$.

Dobijen izlaz se provlači redom nazad na **Euler Integrator** za konačno predskazivanje $\mathbf{x}_{t+1}$.


In [2]:
# Brza inicijalizacija parametara i težina fine-tuned Waterdrop + NANS WCSPH modela
CONFIG_PATH = 'configs/wcsph_transfer.yaml'
CHECKPOINT_PATH = 'checkpoints/best_model_wcsph.pt'

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# loadujemo weights
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

# Kreiranje instance naseg projektovanog modela
model = GNNSimulator.from_config(config).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

broj_parametara = sum(p.numel() for p in model.parameters())
print(f"GNN Arhitektura je inicijalizovana.")
print(f"Veličina modela: {broj_parametara:,} ukupno utreniranih parametara.")


GNN Arhitektura je inicijalizovana.
Veličina modela: 1,591,954 ukupno utreniranih parametara.


## 5. Rezultati i Vizuelizacija GNN modela (Rollout)

Da bismo testirali efektivnost mašinskog učenja u nelinearnom haosu fluida sa neuređenim parametrima stišljivosti, izvršavamo takozvani **Rollout**. Rollout predstavlja prepuštanje generativnih odluka isključivo mašinskom modulu na vremenski red stotina iteracija. Prilikom obimnog Rollout-a kod postavki od preko 4,000 čestica, standardnim mašinama potrebno je više od 10 minuta. Zato u prezentaciji prikazujemo smanjenu trajektoriju tečnosti (od 1,500 čestica) koja obogaćuje interaktivnost ove radne površine i pruža uvid u proračunsku vizualizaciju iz simulacije!


In [3]:
# Priprema domena i pre-generisanje Ground Truth numeričke simulacije, smanjenog uzorka (1500 čestica, 150 timestepova)
rng = np.random.RandomState(42)  # Fiksiran seed za smislen prikaz
num_particles = 1500

# Pokretanje našeg CPython integrisanog koda na račun fizike 
sim = FluidSimulation(
    num_particles=num_particles,
    gravitational_constant=9.81,
    softening_length=0.015,
    integrator='symplectic_euler',
    position_scale=0.8,
    rest_density=1000.0,
    stiffness=1000.0,
    exponent=7.0,
    viscosity=0.3
)

# Nasumični kapljivi početak sfere fluida unutar okvira od 0.1 do 0.9 dimenzija prostora
cx, cy = rng.uniform(0.3, 0.5), rng.uniform(0.4, 0.6)
sim.initialize_random_state(position_scale=0.8, velocity_scale=0.0, mass_range=(1.0, 1.0), start=(cx, cy))

print("1. Izvršavanje ground-truth WCSPH referentne simulacije SPH čestica...")
positions, velocities, accelerations = sim.simulate(
    total_time=150 * 5 * 0.0005,
    dt=0.0005,
    save_every=5,
    quiet=True
)

# Preoblikovanje parametara za GNN
gt_positions = (positions + 0.1).astype(np.float32)
gt_velocities = (gt_positions[1:] - gt_positions[:-1]).astype(np.float32)
print("Generisan uzorak klasične simulacije.")


1. Izvršavanje ground-truth WCSPH referentne simulacije SPH čestica...


Generisan uzorak klasične simulacije.


In [4]:
from tqdm.notebook import tqdm

print("2. Predikcije GNN Simulatora - Faza Autoregresivnog Rollout-a (učenje iz proslosti)...")

history_window = 5 # 5 sekvencijalnih koraka koje gleda GNN
T = 150 # ukupna animacija
pred_positions = np.zeros_like(gt_positions[:T])
pred_positions[:history_window + 1] = gt_positions[:history_window + 1].copy()

# Konverzija za Torch Device
particle_type_t = torch.full((num_particles,), 5, dtype=torch.int64).to(device)
mass_t = torch.ones(num_particles, dtype=torch.float32).to(device)
dummy_acc_t = torch.zeros(num_particles, 2, dtype=torch.float32, device=device)

vel_history_t = torch.from_numpy(gt_velocities[:history_window]).to(device)
current_pos_t = torch.from_numpy(gt_positions[history_window]).to(device)
current_vel_t = torch.from_numpy(gt_velocities[history_window - 1]).to(device)

# Sekvencijalna Forward inferenca modela!
with torch.no_grad():
    for t in tqdm(range(history_window, T - 1), desc='Stanje Mreže'):
        vel_flat_t = vel_history_t.transpose(0, 1).reshape(num_particles, history_window * 2)
        
        # Oformljuje radijus graf na bazi susedstva čestica preko CPU/GPU-a
        graph = build_graph(
            positions=current_pos_t, velocities=vel_flat_t, particle_type=particle_type_t,
            masses=mass_t, target_acc=dummy_acc_t, connectivity_radius=0.015, bounds=[[0.1, 0.9], [0.1, 0.9]]
        )
        
        predicted_acc_t = model(graph)
        new_vel_t = current_vel_t + predicted_acc_t
        new_pos_t = current_pos_t + new_vel_t
        
        pred_positions[t + 1] = new_pos_t.cpu().numpy()
        vel_history_t = torch.cat([vel_history_t[1:], new_vel_t.unsqueeze(0)], dim=0)
        current_pos_t = new_pos_t
        current_vel_t = new_vel_t

# Preciznost nad vremenom: Mean Squared Error (Srednja kvadratna greska pozicije)
step_mse = np.mean((pred_positions[history_window+1:] - gt_positions[history_window+1:T])**2, axis=(1, 2))
print(f"Srednja kvadratna greška predikcije u odnosu na izvorni algoritam (Rollout MSE): {step_mse.mean():.6f}")


2. Predikcije GNN Simulatora - Faza Autoregresivnog Rollout-a (učenje iz proslosti)...


Stanje Mreže:   0%|          | 0/144 [00:00<?, ?it/s]

Srednja kvadratna greška predikcije u odnosu na izvorni algoritam (Rollout MSE): 0.000013


In [5]:
# Vizualizacija - Uporedni prikaz istinitih i predvidivih položaja čestica (Animacija)
bounds = [[0.1, 0.9], [0.1, 0.9]]
x_lim = bounds[0]
y_lim = bounds[1]
N = num_particles
sim_dt = 0.0025 # Efektivni dt (save_every * dt = 5 * 0.0005)

# Animacija na 60fps sa tacnim vremenom simulacije
TARGET_FPS = 60
sim_fps = 1.0 / sim_dt
frame_skip = max(1, int(sim_fps / TARGET_FPS))
frames = list(range(0, T, frame_skip))
interval_ms = 1000.0 / TARGET_FPS

print(f"Brzina animacije: {frame_skip} skip")

fig, (ax_gt, ax_pred) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('WCSPH Rollout: Ground Truth vs GNN Predikcija', fontsize=14, fontweight='bold')

for ax, title in [(ax_gt, 'Ground Truth (WCSPH)'), (ax_pred, 'GNN Predikcija')]:
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_facecolor('#ffffff')
    ax.set_xticks([])
    ax.set_yticks([])

from matplotlib.colors import LinearSegmentedColormap
water_colormap = LinearSegmentedColormap.from_list('ocean_blue', ['#0088cc', '#000033'])
water_colormap_gnn = LinearSegmentedColormap.from_list('ocean_purple', ['#e175ff', '#5a0778'])
particle_colors = np.arange(N)

scatter_gt = ax_gt.scatter(gt_positions[0, :, 0], gt_positions[0, :, 1], s=4, c=particle_colors, cmap=water_colormap, alpha=0.9)
scatter_pred = ax_pred.scatter(pred_positions[0, :, 0], pred_positions[0, :, 1], s=4, c=particle_colors, cmap=water_colormap_gnn, alpha=0.9)
time_text = fig.text(0.5, 0.02, '', ha='center', fontsize=11)

def init():
    scatter_gt.set_offsets(gt_positions[0, :, :2])
    scatter_pred.set_offsets(pred_positions[0, :, :2])
    return scatter_gt, scatter_pred, time_text

def animate(frame_idx):
    t = frames[frame_idx]
    scatter_gt.set_offsets(gt_positions[t, :, :2])
    scatter_pred.set_offsets(pred_positions[t, :, :2])
    time_text.set_text(f't = {t * sim_dt:.3f}s  (Korak {t}/{T-1})')
    return scatter_gt, scatter_pred, time_text

anim = animation.FuncAnimation(fig, animate, init_func=init,
                               frames=len(frames), interval=interval_ms, blit=True)
plt.close(fig)
HTML(anim.to_html5_video())


Brzina animacije: 6 skip


## 6. Zaključak kursnog projekta

Ovaj uvid u polje simulacije tečnosti ističe moć prožimanja strogih fizičkih zakona predviđenih matematičkom jednačinom sa moćnom generalizacijom dubokih neuronskih mreža.
Izvršenjem **Symplectic Euler** bazirane integracije i finim tuning-om (*Transfer Learning*) DeepMind struktura, kreirana arhitektura uči iteriranje dinamike složenog nelinearnog sveta pri izvesnom broju čestica bez žrtvovanja stabilnosti - otvarajući vrata bržim i skalabilnijim kompjuterskim grafikama sutrašnjice.
